# Resipsc V3 - Free Kaggle Training

**Settings:** GPU P100 | Internet: ON | 30 hr limit

**Steps:**
1. Click **Run All** (▶ button above)
2. Wait ~3 hours
3. Download `v3_model.zip` from Output panel (right side)
4. Run inference locally on your laptop

In [ ]:
# 1. INSTALL DEPENDENCIES
!pip install -q scvi-tools==1.5.2 scanpy==1.12.4 torch==2.1.0 pytorch-lightning==2.1.0 anndata==0.10.0 zarr==2.15.0 2>&1 | tail -5

In [ ]:
# 2. GET CODE FROM GITHUB
!git clone https://github.com/227182038-rgb/Resipsc-2.git /kaggle/working/resipsc-v3 2>&1 | tail -3
%cd /kaggle/working/resipsc-v3

In [ ]:
# 3. DOWNLOAD CZ CELLxGENE CENSUS DATA (~1 GB, 5-10 min)
import sys, os
sys.path.insert(0, '.')
from resipsc.data.real_acquisition import prepare_training_corpus

print("=" * 60)
print("DOWNLOADING CZ CELLxGENE CENSUS DATA")
print("=" * 60)
corpus, panel = prepare_training_corpus('data/real')
print(f"✅ Downloaded: {corpus.n_obs:,} cells x {corpus.n_vars} genes")
print(f"   PSC fraction: {corpus.obs['is_psc'].mean():.2%}")

In [ ]:
# 4. TRAIN V3 MODEL (100 epochs ~2.5 hrs on P100)
from resipsc.training.v3_pipeline import train_v3_complete

print("=" * 60)
print("TRAINING V3 MODEL (scVI + Gating + EVT)")
print("=" * 60)

result = train_v3_complete(
    data_dir='data/real',
    output_dir='/kaggle/working/models/v3_kaggle',
    device='cuda',
    max_epochs=100,
    batch_size=256
)

print("=" * 60)
print("✅ TRAINING COMPLETE!")
print("=" * 60)
print(f"Keys: {list(result.keys())}")
print(f"Latent shape: {result['latent'].shape}")

In [ ]:
# 5. EXPORT MODEL FOR DOWNLOAD
import zipfile, os
zip_path = '/kaggle/working/v3_model.zip'
model_dir = '/kaggle/working/models/v3_kaggle'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(model_dir):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, '/kaggle/working/models')
            zf.write(filepath, arcname)

print(f"✅ Model packaged: {zip_path}")
print(f"   Size: {os.path.getsize(zip_path) / 1e6:.1f} MB")
print("=" * 60)
print("📥 DOWNLOAD 'v3_model.zip' FROM KAGGLE OUTPUT PANEL (right side)")
print("=" * 60)